In [1]:
!pip install pybounds

DEPRECATION: Loading egg at c:\users\mayc06\appdata\local\anaconda3\envs\celliniwindsim\lib\site-packages\flyplotlib-0.0.1-py3.11.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation.. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at c:\users\mayc06\appdata\local\anaconda3\envs\celliniwindsim\lib\site-packages\pychebfun-0.3-py3.11.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation.. Discussion can be found at https://github.com/pypa/pip/issues/12330


In [1]:
import time
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import scipy
import pynumdiff
import warnings
warnings.filterwarnings('ignore')

from math import ceil,floor
from pybounds import Simulator#, SlidingEmpiricalObservabilityMatrix, FisherObservability, SlidingFisherObservability, ObservabilityMatrixImage, colorline
from scipy.interpolate import make_interp_spline
from PFN_model import PFN, BuildSSInitial
import utils
from utils import unwrap_angle, wrapToPi
from extract_trajectories_from_orcoflashStupski import get_trajlist_from_behdata


# Define system dynamics and measurements
This example uses a model of an insect flying in the presence of wind.

See the following reference for details:

Floris van Breugel
A Nonlinear Observability Analysis of Ambient Wind Estimation with Uncalibrated Sensors, Inspired by Insect Neural Encoding
2021 60th IEEE Conference on Decision and Control (CDC)
DOI: 10.1109/CDC45484.2021.9683219

The system dynamics are described by seven primary states:
* altitude $z$
* parallel velocity $v_{\parallel}$
* perpendicular velocity $v_{\perp}$
* heading $\phi$
* angular velocity $\dot{\phi}$
* wind speed $w$
* wind direction $\zeta$

And the system dynamics are given by
$$
\dot{\mathbf{x}} = \begin{bmatrix} \dot{z} \\ \dot{v}_{\parallel} \\ \dot{v}_{\perp} \\ \dot{\phi} \\ \ddot{\phi} \\ \dot{w}  \\ \dot{\zeta} \end{bmatrix} =
f(\mathbf{x}) = \begin{bmatrix}
\dot{z} \\
\frac{1}{m}(k_{m_1}u_{\parallel} - C_{\parallel} a_{\parallel}) + v_{\perp} \dot{\phi} \\
 \frac{1}{m}(k_{m_3}u_{\perp} - C_{\perp} a_{\perp}) - v_{\parallel} \dot{\phi} \\
  \dot{\phi} \\
   \frac{1}{I}(k_{m_4}u_{\phi} - C_{\phi} \dot{\phi} + k_{m_2} u_{\perp}) \\
    \dot{w} \\
     \dot{\zeta} \\
\end{bmatrix}
$$

where the air velocity is given by

$$
\begin{bmatrix} a_{\parallel} \\ a_{\perp} \end{bmatrix} =  \begin{bmatrix} v_{\parallel} - w \cos(\phi - \zeta) \\ v_{\perp} + w \sin(\phi - \zeta) \end{bmatrix}
$$
The inputs $u_{\bullet}$ are
* parallel thrust force $u_{\parallel}$
* perpendicular thrust force $u_{\perp}$
* turning torque $u_{\phi}$

The inertia parameters (mass $m$ and inertia $I$), damping terms $C_{\bullet}$, and motor calibration coefficients $k_{m_{\bullet}}$ can also be considered states. Other auxiliary states, like the $x$ and $y$ position can also be added.

The putative system measurements are:
* heading $\phi$
* ground speed angle $\psi$
* apparent airflow angle $\gamma$
* apparent airflow magnitude $a$
* ground speed magnitude $g$
* optic flow $g/z$

Where the measurement function is given by:

$$
\mathbf{y} = h(\mathbf{x}) = \begin{bmatrix} \phi \\ \psi \\ \gamma  \\ a \\ g \\ r \end{bmatrix} =
\begin{bmatrix} \phi \\
\arctan(v_{\perp}/ v_{\parallel}) \\
\arctan(a_{\perp} / a_{\parallel}) \\
\sqrt{a_{\parallel}^2 + a_{\perp}^2} \\
\sqrt{v_{\parallel}^2 + v_{\perp}^2} \\
\sqrt{v_{\parallel}^2 + v_{\perp}^2} / z \\
\end{bmatrix}
$$



## Define dynamics function
The dynamics function takes in a list of states $X$ and a list of inputs $U$ and outputs the derivative of the states.

The optional state & input names must be in the same order as the states & inputs in $X$ & $U$.

In [2]:
state_names = ['x',  # x position [m]
               'y',  # y position [m]
               'z',  # z position (altitude) [m]
               'v_para',  # parallel velocity [m/s]
               'v_perp',  # perpendicular velocity [m/s]
               'phi', # heading [rad]
               'phi_dot',  # angular velocity [rad/s]
               'w',  # ambient wind speed [m/s]
               'zeta',  # ambient wind angle [rad]
               'm', # mass [kg]
               'I',  # inertia [kg*m^2]
               'C_para',  # parallel damping [N*s/m]
               'C_perp',  # perpendicular damping [N*s/m]
               'C_phi',  # rotational damping [N·m/rad/s]
               'km1',  # parallel motor calibration coefficient
               'km2',  # offset motor calibration coefficient
               'km3',  # perpendicular motor calibration coefficient
               'km4',  # rotational motor calibration coefficient
               ]

input_names = ['u_para',  # parallel thrust force [N]
               'u_perp',  # perpendicular thrust force [N]
               'u_phi'#,  # torque [N*m]
               #'u_w',                                                            ####
               #'u_zeta'                                                          ####
               ]
def f(X, U):
    # States
    x, y, z, v_para, v_perp, phi, phi_dot, w, zeta, m, I, C_para, C_perp, C_phi, km1, km2, km3, km4 = X

    # Control Inputs
    u_para, u_perp, u_phi = U
    #u_para, u_perp, u_phi, u_w, u_zeta = U                                       ####

    # Air velocity
    a_para = v_para - w * np.cos(phi - zeta)
    a_perp = v_perp + w * np.sin(phi - zeta)

    # Acceleration
    v_para_dot = ((km1 * u_para - C_para * a_para) / m) + (v_perp * phi_dot)
    v_perp_dot = ((km3 * u_perp - C_perp * a_perp) / m) - (v_para * phi_dot)

    # Angular acceleration
    phi_ddot = (km4 * u_phi / I) - (C_phi * phi_dot / I) + (km2 * u_para / I)

    # Other dynamics
    x_dot = v_para * np.cos(phi) - v_perp * np.sin(phi)
    y_dot = v_para * np.sin(phi) + v_perp * np.cos(phi)
    z_dot = 0*x
    w_dot = 0*x                                                                ####
    #w_dot = w * u_w
    zeta_dot = 0*x                                                             ####
    #zeta_dot = zeta * u_zeta
    m_dot = 0*x
    I_dot = 0*x
    C_para_dot = 0*x
    C_perp_dot = 0*x
    C_phi_dot = 0*x
    km1 = 0*x
    km2 = 0*x
    km3 = 0*x
    km4 = 0*x

    # Package and return xdot
    x_dot = [x_dot, y_dot, z_dot, v_para_dot, v_perp_dot, phi_dot, phi_ddot, w_dot, zeta_dot, m_dot, I_dot, C_para_dot, C_perp_dot, C_phi_dot, km1, km2, km3, km4]

    return x_dot

## Define measurement function
The measurement function takes in a list of states $X$ and a list of inputs $U$ and outputs the measurements $Y$.

The optional measurement names must be in the same order as the measurements in $Y$.

In [3]:
measurement_names = ['phi', 'psi', 'gamma', 'a', 'g', 'r']
def h(X, U):
    # States
    x, y, z, v_para, v_perp, phi, phi_dot, w, zeta, m, I, C_para, C_perp, C_phi, km1, km2, km3, km4 = X

    # Inputs
    u_para, u_perp, u_phi = U
    #u_para, u_perp, u_phi, u_w, u_zeta = U                                     ####

    # Air velocity ### Is this in world coordinates?
    a_para = v_para - w * np.cos(phi - zeta)  # in fly ref frame
    a_perp = v_perp + w * np.sin(phi - zeta)  # in fly ref frame
    a = np.sqrt(a_para ** 2 + a_perp ** 2)
    gamma = np.arctan2(a_perp, a_para)  # air velocity angle   ### in fly ref frame?

    # Course direction in fly reference frame
    g = np.sqrt(v_para ** 2 + v_perp ** 2)
    psi = np.arctan2(v_perp, v_para)

    # Optic flow
    r = g / z

    # Unwrap angles
    if np.array(phi).ndim > 0:
        if np.array(phi).shape[0] > 1:
            phi = np.unwrap(phi)
            psi = np.unwrap(psi)
            gamma = np.unwrap(gamma)

    # Measurements
    Y  = [phi, psi, gamma, a, g, r]

    # Return measurement
    return Y


# Set up model predictive control

In [4]:
# Parameters in SI units
m = 0.25e-6  # [kg]
I = 5.2e-13  # [N*m*s^2] yaw mass moment of inertia: 10.1242/jeb.02369
# I = 4.971e-12  # [N*m*s^2] yaw mass moment of inertia: 10.1242/jeb.038778
C_phi = 27.36e-12  # [N*m*s] yaw damping: 10.1242/jeb.038778
C_para = m / 0.170  # [N*s/m] calculate using the mass and time constant reported in 10.1242/jeb.098665
C_perp = C_para  # assume same as C_para

# Scale Parameters
m = m * 1e6  # [mg]
I = I * 1e6 * (1e3) ** 2  # [mg*mm/s^2 * mm*s^2]
C_phi = C_phi * 1e6 * (1e3) ** 2  # [mg*mm/s^2 *m*s]
C_para = C_para * 1e6  # [mg/s]
C_perp = C_perp * 1e6  # [mg/s]

## Setup for heading predictor model

In [5]:
from keras.models import load_model
from heading_predictor_functions import augment_with_time_delay_embedding, predict_heading_from_fly_trajectory

## Heading Predictor setup.
predictor_input_names = [
    'groundspeed',
    'groundspeed_angle',
    'airspeed',
    'airspeed_angle',
    'thrust',
    'thrust_angle',
]

predictor_output_names = ['heading_angle_x', 'heading_angle_y']

time_window = 4

time_augmentation_kwargs = {
    "time_window": time_window,
    "input_names": predictor_input_names,
    "output_names": predictor_output_names,
    "direction": "backward"
}

headingPredictor=load_model('model_CEM_all-angle-rotate.keras')

# Set trajectory parameters

### For random-parameter datasets

In [6]:
# For RANDOM trajectory parameters (test generalization)

n_traj=5000

rng = np.random.default_rng(29464)

# Set time
fs = 500
dt = 1 / fs
T = 0.1
tsim = np.arange(0.0, T + 0.9*dt, dt)
# tsim = tsim[1:]

# make up a trajectory & wind

phi_0 = rng.random((n_traj,))*2*np.pi


w_0 = rng.random((n_traj,))


zeta_0 = rng.random((n_traj,))*2*np.pi


deltaPhi = np.hstack((rng.random((int(n_traj/2),))*np.pi,rng.random((int(n_traj/2),))*-np.pi))
# deltaPhi = [0.0]                                                                # for no turn

elev_0 = rng.random((n_traj,))*2


v_para_0 = rng.random((n_traj,)) # saccade initial speed m/s
# v_para_0 = w_0
# v_para_dot_0 = [0.0] # m/s^2                                                 # for saccadelike
v_para_dot_0 = rng.random((n_traj,))*100                                        # for no turn


v_perp_0 = 0.0
v_perp_dot_0 = 0.0
v_perp = v_perp_0*np.ones_like(tsim) + v_perp_dot_0*tsim

# Calculate the dynamics of the trajectories

#### If need be, close previously run saving files.

In [8]:
stimfile.close()
PFNfile.close()

### For randomized parameter trajectories

In [7]:
# For RANDOM trajectory parameters - calculate dynamics of the trajectory
stimulus = pd.DataFrame()
PFNdf = pd.DataFrame()

stimfilename = 'n5000rand_withHeadingPredictor_yesdownsample_fixedFlyRef1_fixedPFNpc_noheading_stims'
stimfilename = stimfilename+'.csv'
stimfile = open(stimfilename, 'a')

PFNfilename = 'ANN-PFNinputs_n5000rand_withHeadingPredictor_yesdownsample_fixedFlyRef1_fixedPFNpc_noheading'
PFNfilename = PFNfilename+'.csv'
PFNfile = open(PFNfilename, 'a')

count=0
for i in range(n_traj):
#for i in range(count,n_traj):
    
    simulator = Simulator(f, h, dt=dt, 
                          state_names=state_names, 
                          input_names=input_names, 
                          measurement_names=measurement_names)
    
    w = w_0[i]*np.ones_like(tsim)
    zeta = zeta_0[i]*np.ones_like(tsim)

    vpara_start = v_para_0[i]*np.ones_like(tsim) #+ v_para_dot_0*tsim
    accel = v_para_dot_0[i]*np.square(tsim[0:ceil(len(tsim)/2)])
    v_para = vpara_start - np.append(accel,np.flip(accel[:-1]))
    # v_para = v_para_0*np.ones_like(tsim) # no decel
        
    phi_start = phi_0[i]*np.ones_like(tsim)
    angaccel = (deltaPhi[i])/(1+np.exp(-100*(tsim-0.052)))       # multiply by 0 to keep random order
    phi = phi_start + angaccel                
    # phi = s*np.ones_like(tsim) + (10*d)*tsim # constant turn
    
    alt = elev_0[i]*np.ones_like(tsim)
    
    setpoint_CEM = {'x': 0.0 * np.ones_like(tsim),
                    'y': 0.0 * np.ones_like(tsim),
                    'z': alt,
                    'v_para': v_para,
                    'v_perp': v_perp,
                    'phi': phi,
                    'phi_dot': 0.0*np.ones_like(tsim),
                    'w': 0.0*np.ones_like(tsim),         ###
                    'zeta': 0.0*np.ones_like(tsim),      ###
                    'm': m * np.ones_like(tsim),
                    'I': I * np.ones_like(tsim),
                    'C_para': C_para * np.ones_like(tsim),
                    'C_perp': C_perp * np.ones_like(tsim),
                    'C_phi': C_phi * np.ones_like(tsim),
                    'km1': 1.0 * np.ones_like(tsim),
                    'km2': 0.0 * np.ones_like(tsim),
                    'km3': 1.0 * np.ones_like(tsim),
                    'km4': 1.0 * np.ones_like(tsim),
                   }

    # Update the simulator set-point
    simulator.update_dict(setpoint_CEM, name='setpoint')

    # Define cost function: penalize the squared error between parallel & perpendicular velocity and heading
    cost = ((simulator.model.x['v_para'] - simulator.model.tvp['v_para_set']) ** 2 +
            (simulator.model.x['v_perp'] - simulator.model.tvp['v_perp_set']) ** 2 +
            (simulator.model.x['phi'] - simulator.model.tvp['phi_set']) ** 2 #+
            #(simulator.model.x['w'] - simulator.model.tvp['w_set']) ** 2 + 
            #(simulator.model.x['zeta'] - simulator.model.tvp['zeta_set']) ** 2
           )

    # Set cost function
    simulator.mpc.set_objective(mterm=cost, lterm=cost)

    # Set input penalty: make this small for accurate state following
    simulator.mpc.set_rterm(u_para=1e-6, u_perp=1e-6, u_phi=1e-6)

    t_sim1, x_sim1, u_sim, y_sim1 = simulator.simulate(x0=None, mpc=True, return_full_output=True)

    t_sim, x_sim, u_sim2, y_sim = simulator.simulate(x0={'w':w[0],'zeta':zeta[0]}, mpc=False, u=u_sim, return_full_output=True)

    trajec = pd.DataFrame()
    trajec['timestamp'] = tsim
    trajec['groundspeed'] = y_sim['g']
    trajec['groundspeed_angle'] = y_sim['psi']+y_sim['phi']
    trajec['thrust'] = np.sqrt(u_sim['u_para']**2 + u_sim['u_perp']**2)
    trajec['thrust_angle'] = np.arctan2(u_sim['u_perp'],u_sim['u_para'])
    trajec['airspeed'] = y_sim['a']
    trajec['airspeed_angle'] = y_sim['gamma']+y_sim['phi']
    trajec['position_x'] = x_sim['x']
    trajec['position_y'] = x_sim['y']
    trajec['heading_angle_x'] = np.cos(y_sim['phi'])
    trajec['heading_angle_y'] = np.sin(y_sim['phi'])


    # To downsample CEM data
    trajec = trajec.loc[np.arange(0,len(t_sim1),int(0.01/0.002))]
    trajec.reset_index(drop=True,inplace=True)

    heading_angle_predicted = predict_heading_from_fly_trajectory(trajec, 
                                                                  24,
                                                                  augment_with_time_delay_embedding, 
                                                                  headingPredictor, smooth=True,
                                                                  **time_augmentation_kwargs)

    head_pred_unwrap = utils.unwrap_angle(heading_angle_predicted)

    stimulus['time'] = tsim
    stimulus['obj_id'] = i #count
    stimulus['heading'] = np.nan
    stimulus['heading'].loc[np.arange(0,len(t_sim1),int(0.01/0.002))] = np.round(head_pred_unwrap,6)
    stimulus['heading'].interpolate(method='cubic',inplace=True)
    stimulus['course_dir'] = np.round(y_sim['phi'],6)
    stimulus['groundspeed_angle'] = np.round(y_sim['psi'] + y_sim['phi'],6)
    stimulus['airspeed'] = np.round(y_sim['a'],6)
    stimulus['gamma'] = np.round(y_sim['gamma'] + y_sim['phi'],6) - stimulus['heading']
    stimulus['gspd'] = np.round(y_sim['r'],6)
    stimulus['psi'] = np.round(y_sim['psi'] + y_sim['phi'],6) - stimulus['heading']
    stimulus['zeta'] = np.round(x_sim['zeta'],6)
    stimulus['wspd'] = np.round(x_sim['w'],6)
    stimulus['altitude'] = np.round(x_sim['z'],6)
    stimulus['fspd'] = np.round(y_sim['g'],6)
    stimulus['thrust'] = np.sqrt(u_sim['u_para']**2 + u_sim['u_perp']**2)
    stimulus['thrust_angle'] = np.arctan2(u_sim['u_perp'],u_sim['u_para'])
    stimulus['torque'] = u_sim['u_phi']
    stimulus['xpos'] = np.round(x_sim['x'],6)
    stimulus['ypos'] = np.round(x_sim['y'],6)
    stimulus['vpara_input'] = v_para
    stimulus['vperp_input'] = v_perp
    stimulus['phi_input'] = phi

    if count == 0:
        stimulus.to_csv(stimfile)
    else:
        stimulus.to_csv(stimfile,header=False)


    ## Model PFN activity during trajectory
    pfnmodel = PFN()

    p_d = pfnmodel.model_param['PFNd']
    p_v = pfnmodel.model_param['PFNv']
    p_pc = pfnmodel.model_param['PFNpc']
    p_a = pfnmodel.model_param['PFNa']
    AF = [-stimulus['gamma'].copy().iloc[0], 100*stimulus['airspeed'].copy().iloc[0]]
    OF = [-stimulus['psi'].copy().iloc[0], 100*stimulus['gspd'].copy().iloc[0]]   # "gspd" is actually of speed
    bump = [-stimulus['heading'].copy().iloc[0], -stimulus['gamma'].copy().iloc[0], 100*stimulus['airspeed'].copy().iloc[0]]
    initcond = BuildSSInitial(p_d, p_v, p_pc, p_a, AF, OF, bump)

    pfnmodel.run(tsim=np.array(stimulus['time']),
                  phi=-np.array(stimulus['heading']),
                  a=100*np.array(stimulus['airspeed']),
                  gamma=-np.array(stimulus['gamma']),
                  g=100*np.array(stimulus['gspd']),
                  psi=-np.array(stimulus['psi']),
                  initcond=initcond)

    inputs = {}

    inputs['obj_id'] = i #count
    
    neurons = list(pfnmodel.heatmap.keys())
    labels = {'EPG':['EPG_c1','EPG_c2','EPG_c3','EPG_c4','EPG_c5','EPG_c6','EPG_c7','EPG_c8'],
              'PFNdL_pb':['PFNd_c1','PFNd_c2','PFNd_c3','PFNd_c4','PFNd_c5','PFNd_c6','PFNd_c7','PFNd_c8'],
              'PFNdR_pb':['PFNd_c9','PFNd_c10','PFNd_c11','PFNd_c12','PFNd_c13','PFNd_c14','PFNd_c15','PFNd_c16'],
              'PFNvL_pb':['PFNv_c1','PFNv_c2','PFNv_c3','PFNv_c4','PFNv_c5','PFNv_c6','PFNv_c7','PFNv_c8'],
              'PFNvR_pb':['PFNv_c9','PFNv_c10','PFNv_c11','PFNv_c12','PFNv_c13','PFNv_c14','PFNv_c15','PFNv_c16'],
              'PFNpcL_pb':['PFNpc_c1','PFNpc_c2','PFNpc_c3','PFNpc_c4','PFNpc_c5','PFNpc_c6','PFNpc_c7','PFNpc_c8'],
              'PFNpcR_pb':['PFNpc_c9','PFNpc_c10','PFNpc_c11','PFNpc_c12','PFNpc_c13','PFNpc_c14','PFNpc_c15','PFNpc_c16'],
              'PFNaL_pb':['PFNa_c1','PFNa_c2','PFNa_c3','PFNa_c4','PFNa_c5','PFNa_c6','PFNa_c7','PFNa_c8'],
              'PFNaR_pb':['PFNa_c9','PFNa_c10','PFNa_c11','PFNa_c12','PFNa_c13','PFNa_c14','PFNa_c15','PFNa_c16']   
              }
    for celltype in neurons:
        j = 0
        for label in labels[celltype]:
            inputs[label] = pfnmodel.heatmap[celltype][:,j]
            j+=1

    angle_map = np.arange(-np.pi,np.pi,step=np.pi/4)
    n_angles = angle_map.shape[0]
    n_sim = stimulus['time'].shape[0]

    inputs['wind_c1'] = np.empty((n_sim,))
    inputs['wind_c2'] = np.empty((n_sim,))
    inputs['wind_c3'] = np.empty((n_sim,))
    inputs['wind_c4'] = np.empty((n_sim,))
    inputs['wind_c5'] = np.empty((n_sim,))
    inputs['wind_c6'] = np.empty((n_sim,))
    inputs['wind_c7'] = np.empty((n_sim,))
    inputs['wind_c8'] = np.empty((n_sim,))

    for pt in range(n_sim):
        wind = 0.5 + 0.5 * np.cos(angle_map - stimulus['zeta'].iloc[pt] - np.pi)
        inputs['wind_c1'][pt]=wind[0]
        inputs['wind_c2'][pt]=wind[1]
        inputs['wind_c3'][pt]=wind[2]
        inputs['wind_c4'][pt]=wind[3]
        inputs['wind_c5'][pt]=wind[4]
        inputs['wind_c6'][pt]=wind[5]
        inputs['wind_c7'][pt]=wind[6]
        inputs['wind_c8'][pt]=wind[7]

    PFNdf = pd.DataFrame(inputs)
    if count == 0:
        PFNdf.to_csv(PFNfile)
    elif count!=0:
        PFNdf.to_csv(PFNfile,header=False)

    print('windspd:'+str(stimulus['wspd'].iloc[0]))

    print(str(count+1)+' out of '+str(n_traj))

    count+=1                   
                        
stimfile.close()
PFNfile.close()

24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step
windspd:0.733661
1 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
windspd:0.59956
2 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
windspd:0.254013
3 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
windspd:0.477309
4 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
windspd:0.738448
5 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
windspd:0.408662
6 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
windspd:0.879758
7 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
windspd:0.799358
8 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
windspd:0.814902
9 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
windspd:0.991516
10 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
windspd:0.137032
11 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
windspd:0.304421
12 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
windspd:0.347388
13 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
windspd:0.284659
14 

24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
windspd:0.760525
78 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
windspd:0.802765
79 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
windspd:0.14429
80 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
windspd:0.843102
81 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
windspd:0.106154
82 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
windspd:0.708221
83 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
windspd:0.203249
84 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
windspd:0.489307
85 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
windspd:0.114873
86 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
windspd:0.995008
87 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
windspd:0.438473
88 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
windspd:0.292509
89 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
windspd:0.838449
90 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
windspd:0.3

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
windspd:0.782254
154 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
windspd:0.289395
155 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
windspd:0.436484
156 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
windspd:0.796006
157 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
windspd:0.344182
158 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
windspd:0.561288
159 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
windspd:0.205886
160 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
windspd:0.862832
161 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
windspd:0.186969
162 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
windspd:0.259056
163 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
windspd:0.490166
164 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
windspd:0.395132
165 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
windspd:0.442449
166 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step


24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
windspd:0.267528
230 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
windspd:0.646008
231 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
windspd:0.808226
232 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
windspd:0.598381
233 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
windspd:0.960255
234 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
windspd:0.609188
235 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
windspd:0.636733
236 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
windspd:0.103422
237 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
windspd:0.044631
238 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
windspd:0.160157
239 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
windspd:0.418358
240 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
windspd:0.737221
241 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
windspd:0.643985
242 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/st

24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
windspd:0.216955
306 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
windspd:0.286775
307 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
windspd:0.023503
308 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
windspd:0.57656
309 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
windspd:0.955106
310 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
windspd:0.572046
311 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
windspd:0.312778
312 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
windspd:0.490916
313 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
windspd:0.395064
314 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
windspd:0.540322
315 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
windspd:0.514026
316 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
windspd:0.758284
317 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
windspd:0.586311
318 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/ste

...

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
windspd:0.695334
4665 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
windspd:0.0213
4666 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
windspd:0.987451
4667 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
windspd:0.072691
4668 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
windspd:0.471932
4669 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
windspd:0.766024
4670 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
windspd:0.798679
4671 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
windspd:0.458066
4672 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
windspd:0.462035
4673 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
windspd:0.573555
4674 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
windspd:0.044044
4675 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
windspd:0.149231
4676 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
windspd:0.50867
4677 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
windspd:0.75842
4740 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
windspd:0.038167
4741 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
windspd:0.817803
4742 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
windspd:0.304345
4743 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
windspd:0.515671
4744 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
windspd:0.908192
4745 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
windspd:0.746579
4746 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
windspd:0.096381
4747 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
windspd:0.892228
4748 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
windspd:0.710432
4749 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
windspd:0.934647
4750 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
windspd:0.974133
4751 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
windspd:0.9351
4752 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
windspd:0.34258
4815 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
windspd:0.526296
4816 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
windspd:0.83321
4817 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
windspd:0.652189
4818 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
windspd:0.54878
4819 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
windspd:0.624405
4820 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
windspd:0.313325
4821 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
windspd:0.526803
4822 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
windspd:0.250534
4823 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
windspd:0.708988
4824 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
windspd:0.384272
4825 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
windspd:0.643749
4826 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
windspd:0.964944
4827 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
windspd:0.61632
4890 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
windspd:0.721976
4891 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
windspd:0.855246
4892 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
windspd:0.66613
4893 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
windspd:0.635374
4894 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
windspd:0.815821
4895 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
windspd:0.043638
4896 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
windspd:0.031013
4897 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
windspd:0.805098
4898 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
windspd:0.52552
4899 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
windspd:0.813996
4900 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
windspd:0.718246
4901 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
windspd:0.354104
4902 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
windspd:0.6255
4965 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
windspd:0.091617
4966 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
windspd:0.37616
4967 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
windspd:0.419779
4968 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
windspd:0.09586
4969 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
windspd:0.214412
4970 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
windspd:0.231436
4971 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
windspd:0.303938
4972 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
windspd:0.198657
4973 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
windspd:0.809396
4974 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
windspd:0.387335
4975 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
windspd:0.195748
4976 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
windspd:0.002431
4977 out of 5000
24
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 3